# Clean NVE rock avalanche events

Same flow as `03_clean_nve_avalanche_events.ipynb`, for
`01_ingestion/nve_rock_avalanche_events.py` (skredType 110-113: rockfall and
rockslide — the largest category, ~54,000 records). Reads `02_data/raw/`
directly — `nve_*` sources are deliberately excluded from `records` (see the
comment in `backend/transform.py`'s `build_tables()`, and
`05_tests/test_records_excludes_nve.py`) — and saves its own table,
`nve_rock_avalanche_events_clean`, never touching `records`.

In [ ]:
import sys

sys.path.insert(0, "..")  # the notebook runs from 03_notebooks/

import pandas as pd

from backend import config, storage

storage.tables()

In [ ]:
raw_path = next(
    (p for p in config.RAW_DIR.glob("nve_rock_avalanche_events.*") if p.suffix.lower() in storage.DATA_SUFFIXES),
    None,
)
raw = storage.read_file(raw_path) if raw_path else pd.DataFrame()
raw.head()

In [ ]:
NORWEGIAN_MONTHS = {
    1: "Januar", 2: "Februar", 3: "Mars", 4: "April", 5: "Mai", 6: "Juni",
    7: "Juli", 8: "August", 9: "September", 10: "Oktober", 11: "November", 12: "Desember",
}
NORWEGIAN_WEEKDAYS = {
    0: "Mandag", 1: "Tirsdag", 2: "Onsdag", 3: "Torsdag", 4: "Fredag", 5: "Lørdag", 6: "Søndag",
}

clean = pd.DataFrame()

if raw.empty:
    print("No `nve_rock_avalanche_events` raw file yet — run `make ingestion` first.")
else:
    clean = raw.copy()
    clean.columns = clean.columns.str.lower()

    clean["døde"] = clean["døde"].fillna(0).astype("int64")

    clean["dato"] = pd.to_datetime(clean["dato"], format="%Y-%m-%d")
    clean["year"] = clean["dato"].dt.year.astype("Int64")
    clean["month"] = clean["dato"].dt.month.map(NORWEGIAN_MONTHS)
    clean["day"] = clean["dato"].dt.weekday.map(NORWEGIAN_WEEKDAYS)

    clean = clean[clean["latitude"] < 74]

    storage.save("nve_rock_avalanche_events_clean", clean)

clean.head()